# 07 Epoch Benchmarking Hub: per-benchmark scores

**Purpose.** This notebook turns the separate benchmark CSV files (61 of the 88 listed benchmarks have a file) from Epoch's Benchmarking Hub into one tidy long table, `benchmark_scores.csv`, with one row per model version and benchmark. Q1 uses it to show *where* a model is strong (coding, maths, agent tasks) and not only its single ECI number.

**Question served.** Q1, with extra detail for Q2.

**How to run.** Run all cells from top to bottom. The archive is downloaded to `raw/`, unpacked and read. If the download is blocked, the notebook uses the copy in `Data/raw/epoch/`.

## At a glance: links, authorisation, download, timing

| Item | Detail |
|---|---|
| Kind | Download of one ZIP archive of CSV files |
| Where it comes from | Epoch AI. Data page https://epoch.ai/data (entry "Capabilities & benchmarking"). Direct file https://epoch.ai/data/benchmark_data.zip . Hub https://epoch.ai/benchmarks |
| Licence and authorisation | Creative Commons Attribution (archive README links CC BY 4.0). Credit Epoch AI and, where a file names one, the original benchmark authors. No account or key. Details in Section 1 |
| Download size | about 2.4 MB |
| Expected time | download 2 to 10 seconds, whole notebook under 30 seconds. Measured in our test run: 2.1 seconds, using the local copy because the test machine could not reach epoch.ai |
| Data as of | release dates of the models in the archive. See `processed/data_currency.csv`. The data page listed the archive as updated 2026-10-17 |
| To update | Run the notebook again |

## 1. Source, reference links and authorisation

| Item | Detail |
|---|---|
| Publisher | Epoch AI |
| Dataset | Benchmarking Hub results: runs by Epoch and results collected from other sources |
| Hub page | https://epoch.ai/benchmarks |
| Chart pages (same project, for comparing by eye) | https://epoch.ai/benchmarks?view=graph&tab=eci shows two charts we can check against: "Epoch Capabilities Index over time" and "Frontier over time for math benchmarks" (71 results on 18 Oct 2026, page footer "CC-BY", no version shown). The math chart plots the best score so far on Mock AIME 24-25, FrontierMath Tiers 1-3, MATH Level 5, ProofBench, FrontierMath Tier 4, GSM8K and FrontierMath Erdős. Our table holds every run and not only the frontier, so counts differ. FrontierMath Erdős has no CSV in the archive and is listed as skipped in the catalogue |
| Download page | https://epoch.ai/data , entry "Capabilities & benchmarking", ZIP |
| Data page | https://epoch.ai/data |
| File used | https://epoch.ai/data/benchmark_data.zip |
| Licence | Creative Commons Attribution. The README inside the archive states: "Epoch AI's data is free to use, distribute, and reproduce provided the source and authors are credited under the Creative Commons Attribution license" and links https://creativecommons.org/licenses/by/4.0/ |
| Authorisation | No account, key or login needed. Public file. No scraping. |
| Citation (from the archive README) | Epoch AI, 'Capabilities & benchmarking'. Published online at epoch.ai. Retrieved from https://epoch.ai/benchmarks [online resource] |
| Earlier snapshot | The same archive was downloaded on 2026-10-14 for the Part 2 app (2,382,498 bytes). A copy is in `Data/raw/epoch/benchmark_data.zip` |
| Note on third-party scores | Some benchmark files collect scores published by other organisations. Epoch's licence covers Epoch's compilation. Cite Epoch and, where the file names a source, the original benchmark authors. |

## 2. Table structure

### 2.1 Structure of the source archive

The ZIP holds one CSV per benchmark and three index files. All files sit at the top level of the archive.

| File | Content |
|---|---|
| `benchmark_metadata.csv` | One row per benchmark. Columns `benchmark`, `in_eci`, `source_file`, `score_column`, `scale` (a multiplier to a 0 to 1 fraction), `random_baseline`, `score_ceiling`, `release_date`, `superseded_by`. This is the map the notebook follows. Benchmarks with an empty `source_file` have no downloadable CSV |
| `model_metadata.csv` | One row per model version. Columns `model_version`, `model_group`, `date`, `display_name`, `organization`, `country`, `accessibility`, `training_compute_flop`. About 1,158 rows, some of them empty |
| `README.md` | Licence and citation |
| `<benchmark>.csv` (about 61 files) | Results for one benchmark. The columns differ from file to file. Common ones are `Model version`, a score column named in `benchmark_metadata.csv`, `Release date`, `Organization`, `Country`, `Training compute (FLOP)` and an error column such as `stderr` or `Accuracy Standard Error` |

### 2.2 Structure of the output `processed/benchmark_scores.csv`

| Output column | Type | How it is made |
|---|---|---|
| `benchmark` | text | from `benchmark_metadata.csv` |
| `in_eci` | bool | whether Epoch uses the benchmark inside ECI |
| `model_version` | text | `Model version` in the benchmark file. This is **not** the same string as `Model` in the ECI table. A crosswalk is needed |
| `score_raw` | float | the column named in `score_column` of the metadata, exactly as in the file |
| `scale_factor` | float | `scale` in the metadata. It is the multiplier that turns the file's units into a 0 to 1 fraction (1 for fractions, 0.01 for percent files, 0.1 for a 0 to 10 file) |
| `score` | float | `score_raw` times `scale_factor`, a 0 to 1 fraction that can be compared across files |
| `score_stderr_raw` | float or empty | the first column whose name holds "stderr", "standard error" or "std err", in the file's own units. Empty if the file has none |
| `random_baseline`, `score_ceiling` | float | from metadata, on the 0 to 1 scale |
| `release_date` | date | `Release date` of the model |
| `organization`, `country` | text | as given in the file |
| `source_file` | text | the CSV the row came from |

### 2.3 Other outputs

| File | Content |
|---|---|
| `benchmark_catalogue.csv` | One row per benchmark: file read or skipped, row count, number of models, number of China-based rows |
| `benchmark_model_metadata.csv` | Copy of `model_metadata.csv` with empty rows removed. Useful for the model crosswalk |

## 3. Set up the paths and download helper

In [1]:
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

T0 = time.time()
FOLDER = "07_epoch_benchmark_scores"
cwd = Path.cwd()
if cwd.name == FOLDER:
    HERE = cwd
elif (cwd / FOLDER).exists():
    HERE = cwd / FOLDER
elif (cwd / "Data" / FOLDER).exists():
    HERE = cwd / "Data" / FOLDER
else:
    raise FileNotFoundError(f"Open this notebook from the folder {FOLDER} (or from Data/). Current folder: {cwd}")

DATA = HERE.parent               # the Data folder
MANUAL = DATA / "raw" / "epoch"                # hand-downloaded or pre-copied source files
RAW = HERE / "raw"               # files this notebook downloads, kept exactly as received
OUT = HERE / "processed"         # tables this notebook writes
for folder in (RAW, OUT):
    folder.mkdir(parents=True, exist_ok=True)
LOG_PATH = RAW / "retrieval_log.json"
print("Notebook folder:", HERE)
print("Manual folder  :", MANUAL)


HEADERS = {"User-Agent": "UTS-36104-A3-student-project (academic use, no contact data stored)"}
TIMEOUT = 120
REFRESH = True   # True: try a fresh download first. False: reuse the file already in raw/ if there is one.


def record(source_id: str, url: str, path: Path, how: str) -> None:
    """Add (or replace) the retrieval-log entry for one raw file."""
    content = path.read_bytes()
    log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
    log[source_id] = {
        "url": url,
        "file": path.name,
        "how": how,
        "retrieved_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest(),
    }
    LOG_PATH.write_text(json.dumps(log, indent=2), encoding="utf-8")
    print(f"logged {source_id}: {len(content):,} bytes via {how}")


def get_file(source_id: str, url: str, target: str, manual_name: str = None) -> Path:
    """Return the path of a raw file. Order: fresh download, earlier download in raw/, manual copy."""
    path = RAW / target
    if REFRESH or not path.exists():
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            path.write_bytes(response.content)
            record(source_id, url, path, "downloaded")
            return path
        except Exception as err:
            print(f"Download of {source_id} failed: {err}")
    manual = MANUAL / (manual_name or target)
    candidates = [p for p in (path, manual) if p.exists()]
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)   # the newest copy wins
        if best == path:
            print(f"Using the earlier download {path.name}")
        else:
            shutil.copy2(best, path)
            record(source_id, url, path, f"copied from manual folder {manual.parent.name}")
        return path
    raise FileNotFoundError(
        f"Could not get {source_id}. Open {url} in a browser, save the file as '{manual_name or target}' "
        f"inside {MANUAL}, then run this cell again.")

import zipfile

Notebook folder: d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\07_epoch_benchmark_scores
Manual folder  : d:\UTS MDSI\36104 (2026)\Assignment 3\Hedging Region AI Model Selection and Data Risk\Data\raw\epoch


## 4. Acquire and unpack the archive

In [2]:
BENCH_URL = "https://epoch.ai/data/benchmark_data.zip"
zip_path = get_file("benchmark_data", BENCH_URL, "benchmark_data.zip")

UNPACKED = RAW / "benchmark_data"
if UNPACKED.exists():
    shutil.rmtree(UNPACKED)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(UNPACKED)
    names = archive.namelist()
print(len(names), "files in the archive")
print(names[:8], "...")

# The index files may sit in a sub-folder if Epoch changes the packaging. Find them wherever they are.
meta_files = list(UNPACKED.rglob("benchmark_metadata.csv"))
assert meta_files, "benchmark_metadata.csv is not in the archive. Epoch changed the packaging."
BASE = meta_files[0].parent
print("Benchmark files are in:", BASE.relative_to(HERE))

logged benchmark_data: 2,401,416 bytes via downloaded
92 files in the archive
['README.md', 'gpqa_diamond.csv', 'math_level_5.csv', 'swe_bench_verified.csv', 'otis_mock_aime_2024_2025.csv', 'frontiermath.csv', 'frontiermath_tier_4.csv', 'frontiermath_tiers_1_3_v2.csv'] ...
Benchmark files are in: raw\benchmark_data


## 5. Inspect the index files

In [3]:
bench_meta = pd.read_csv(BASE / "benchmark_metadata.csv")
print(bench_meta.shape)
print(bench_meta.dtypes)
print()
print("Benchmarks with a CSV file:", bench_meta["source_file"].notna().sum(), "of", len(bench_meta))
print("Benchmarks used inside ECI:", int(bench_meta["in_eci"].astype(str).str.lower().eq("true").sum()))
bench_meta.head(8)

(91, 9)
benchmark           object
in_eci                bool
source_file         object
score_column        object
scale              float64
random_baseline    float64
score_ceiling      float64
release_date        object
superseded_by       object
dtype: object

Benchmarks with a CSV file: 62 of 91
Benchmarks used inside ECI: 62


,benchmark,in_eci,source_file,score_column,scale,random_baseline,score_ceiling,release_date,superseded_by
0,GPQA diamond,True,gpqa_diamond.csv,Best score (across scorers),1.0,0.250,1.00,2023-11-20,NaN
1,FrontierMath-2025-02-28-Private,True,frontiermath.csv,Best score (across scorers),1.0,0.000,0.57,2025-02-28,FrontierMath-Tiers-1-3-v2-Private
2,FrontierMath-Tier-4-2025-07-01-Private,True,frontiermath_tier_4.csv,Best score (across scorers),1.0,0.000,0.60,2025-07-11,FrontierMath-Tier-4-v2-Private
3,FrontierMath-Tiers-1-3-v2-Private,True,frontiermath_tiers_1_3_v2.csv,Best score (across scorers),1.0,0.000,1.00,2026-06-12,NaN
4,FrontierMath-Tier-4-v2-Private,True,frontiermath_tier_4_v2.csv,Best score (across scorers),1.0,0.000,1.00,2026-06-12,NaN
5,MATH level 5,True,math_level_5.csv,Best score (across scorers),1.0,0.000,1.00,2021-03-05,NaN
6,OTIS Mock AIME 2024-2025,True,otis_mock_aime_2024_2025.csv,Best score (across scorers),1.0,0.001,1.00,2024-12-19,NaN
7,SWE-Bench verified,True,swe_bench_verified.csv,Best score (across scorers),1.0,0.000,1.00,2024-08-13,NaN


In [4]:
model_meta = pd.read_csv(BASE / "model_metadata.csv")
model_meta = model_meta.dropna(how="all").reset_index(drop=True)
print(model_meta.shape, "after dropping empty rows")
print(model_meta["country"].value_counts(dropna=False).head(6))
model_meta.to_csv(OUT / "benchmark_model_metadata.csv", index=False, encoding="utf-8")

(1152, 8) after dropping empty rows
country
United States of America    687
China                       274
NaN                         118
France                       54
United Arab Emirates          5
Canada                        3
Name: count, dtype: int64


## 6. Read every benchmark file

For each benchmark with a `source_file` the notebook reads the CSV, takes the score column named in the metadata, finds the error column by name, and appends tidy rows. A file that cannot be read is listed in the catalogue with the reason and does not stop the run.

In [5]:
def find_stderr(columns):
    """Return the first column that looks like a standard error, else None."""
    for col in columns:
        low = col.lower()
        if "stderr" in low or "standard error" in low or "std err" in low or "std. err" in low:
            return col
    return None


frames, catalogue = [], []
for _, row in bench_meta.iterrows():
    entry = {"benchmark": row["benchmark"], "source_file": row["source_file"], "status": "", "rows": 0,
             "models": 0, "china_rows": 0, "score_column": row["score_column"], "stderr_column": None}
    if pd.isna(row["source_file"]):
        entry["status"] = "skipped: no CSV file in the archive"
        catalogue.append(entry)
        continue
    path = BASE / row["source_file"]
    if not path.exists():
        entry["status"] = "skipped: file named in metadata is missing"
        catalogue.append(entry)
        continue
    try:
        data = pd.read_csv(path)
    except Exception as err:
        entry["status"] = f"skipped: cannot read ({err})"
        catalogue.append(entry)
        continue
    score_col = row["score_column"]
    if score_col not in data.columns or "Model version" not in data.columns:
        entry["status"] = "skipped: score or model column not found"
        catalogue.append(entry)
        continue
    err_col = find_stderr(data.columns)
    tidy = pd.DataFrame({
        "benchmark": row["benchmark"],
        "in_eci": str(row["in_eci"]).lower() == "true",
        "model_version": data["Model version"],
        "score_raw": pd.to_numeric(data[score_col], errors="coerce"),
        "scale_factor": pd.to_numeric(row["scale"], errors="coerce"),
        "score_stderr_raw": pd.to_numeric(data[err_col], errors="coerce") if err_col else pd.NA,
        "random_baseline": row["random_baseline"],
        "score_ceiling": row["score_ceiling"],
        "release_date": pd.to_datetime(data["Release date"], errors="coerce") if "Release date" in data.columns else pd.NaT,
        "organization": data["Organization"] if "Organization" in data.columns else pd.NA,
        "country": data["Country"] if "Country" in data.columns else pd.NA,
        "source_file": row["source_file"],
    })
    tidy["score"] = tidy["score_raw"] * tidy["scale_factor"]
    tidy = tidy.dropna(subset=["model_version", "score"])
    entry.update(status="read", rows=len(tidy), models=tidy["model_version"].nunique(),
                 china_rows=int(tidy["country"].eq("China").sum()), stderr_column=err_col)
    catalogue.append(entry)
    frames.append(tidy)

scores = pd.concat(frames, ignore_index=True)
catalogue = pd.DataFrame(catalogue)
print(catalogue["status"].str.split(":").str[0].value_counts())
print()
print("Tidy rows:", len(scores), "| benchmarks:", scores["benchmark"].nunique(), "| model versions:", scores["model_version"].nunique())

status
read       62
skipped    29
Name: count, dtype: int64

Tidy rows: 5251 | benchmarks: 62 | model versions: 910


C:\Users\kaoyi\AppData\Local\Temp\ipykernel_37724\3577264672.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  scores = pd.concat(frames, ignore_index=True)


## 7. Validate

In [6]:
problems = []
if scores.empty:
    problems.append("no rows")
# After applying scale_factor every score should be a fraction between 0 and 1 (small overshoot allowed).
bad_range = scores[(scores["score"] < 0) | (scores["score"] > 1.05)]
share_bad = len(bad_range) / max(len(scores), 1)
print(f"Rows outside 0..1 after scaling: {len(bad_range)} ({share_bad:.1%})")
if share_bad > 0.02:
    problems.append("many scores outside 0..1, check the scale_factor column")
    print(bad_range.groupby("benchmark")["score_raw"].agg(["count", "max"]))
dup = scores.duplicated(subset=["benchmark", "model_version"], keep=False)
print("Rows sharing the same benchmark and model_version:", int(dup.sum()))
unread = catalogue[catalogue["status"].str.startswith("skipped") & catalogue["source_file"].notna()]
print("Files named in metadata but not read:", len(unread))
print(unread[["benchmark", "status"]].to_string(index=False) if len(unread) else "")
print("Problems:", problems if problems else "none")
assert not problems, problems

print()
print("Rows by country (top 5):")
print(scores["country"].value_counts(dropna=False).head(5))

Rows outside 0..1 after scaling: 0 (0.0%)
Rows sharing the same benchmark and model_version: 1138
Files named in metadata but not read: 0

Problems: none

Rows by country (top 5):
country
United States of America    3824
China                        928
NaN                          200
France                       152
United Arab Emirates          92
Name: count, dtype: int64


Rows that share a benchmark and model version are repeated runs or reasoning-effort settings in some files. They are kept so that nothing is thrown away. The dashboard step must decide which run to show (for example the highest score) and say so.

## 8. Save

In [7]:
scores = scores.sort_values(["benchmark", "score"], ascending=[True, False]).reset_index(drop=True)
scores.to_csv(OUT / "benchmark_scores.csv", index=False, encoding="utf-8")
catalogue.to_csv(OUT / "benchmark_catalogue.csv", index=False, encoding="utf-8")
print("saved benchmark_scores.csv", scores.shape)
print("saved benchmark_catalogue.csv", catalogue.shape)

saved benchmark_scores.csv (5251, 13)
saved benchmark_catalogue.csv (91, 8)


## 9. China-based models: where do they have scores?

A quick look at which benchmarks have the most China-based rows. It shows where evidence exists for the hedging region and where it is thin. It is a sanity view and not a result.

In [8]:
coverage = (scores.assign(china=scores["country"].eq("China"))
            .groupby("benchmark")
            .agg(rows=("score", "size"), china_rows=("china", "sum"))
            .sort_values("china_rows", ascending=False))
coverage.head(12)

,rows,china_rows
benchmark,,
GPQA diamond,324,86
OTIS Mock AIME 2024-2025,304,77
DTBench,214,46
LMCA,176,46
WeirdML,173,40
MMLU,217,38
Mystery Game Puzzles,140,37
FrontierMath-Tiers-1-3-v2-Private,120,33
GSM8K,170,32


## How to update, and what to do if the download is blocked

**Automatic.** Run all cells again. Section 4 downloads https://epoch.ai/data/benchmark_data.zip, replaces `raw/benchmark_data.zip` and unpacks it. About 10 seconds.

**Manual download (use this if you see `Download of benchmark_data failed`).**

1. Open https://epoch.ai/data in a browser. Scroll the list "Download our data" to **Capabilities & benchmarking** (ZIP) and click the download arrow at the right end of the row. Do not use the PNG or SVG button under a chart, which only saves a picture.
2. Alternatively paste https://epoch.ai/data/benchmark_data.zip into the address bar.
3. Save it as `Data\raw\epoch\benchmark_data.zip` and **do not unpack it**. The notebook unpacks it.
4. Run the notebook again. With no fresh download, it uses the **newer** of `raw/benchmark_data.zip` and `Data/raw/epoch/benchmark_data.zip`.

**Update check.** After any update, open `raw/retrieval_log.json` in this folder. The `how` field says `downloaded` when the fresh file was used and `copied from manual folder` when the hand-saved copy was used. Then open `processed/data_currency.csv` and confirm `extracted_utc` and `data_covers_to` are what you expect.

## 10. Run time and limits

**Time to extract.** One ZIP of about 2.4 MB, then about 61 small CSV files read from disk. Expect 2 to 10 seconds for the download and 5 to 15 seconds for reading and tidying, so a full run takes **under 30 seconds**. The last cell prints the measured time.

**Limits and known risks.**

1. `model_version` strings (for example `gemini-4-argon_high`) are API-style names with the reasoning setting after an underscore. They do not equal `Model` in notebook 01. The crosswalk step must map them.
2. Score columns differ in meaning (percent correct, win rate, pass@1). `score` is rescaled to a 0 to 1 fraction, but a fraction on one benchmark is not comparable with another without using `random_baseline` and `score_ceiling`. The error column is left in the file's own units.
3. Benchmarks without a CSV in the archive are listed as skipped in the catalogue.
4. China-based coverage is uneven. See the last table.
5. The archive layout is not a fixed schema. If Epoch renames a column, Section 6 marks the file as skipped and the catalogue says why.

## 11. References

1. Epoch AI, Benchmarking Hub. https://epoch.ai/benchmarks
2. Epoch AI, Data page and licence statement. https://epoch.ai/data
3. Epoch AI, benchmark archive. https://epoch.ai/data/benchmark_data.zip
4. Creative Commons, Attribution 4.0 International. https://creativecommons.org/licenses/by/4.0/
5. Project note on licences: `05 Gemini Deep Research/Licence Check for Open-Weight Models (English and 繁體中文).md`

## Data currency: how up to date is this data?

Every notebook writes `processed/data_currency.csv` with the day it was extracted, the dates the data itself covers, and what we know about when the source started and how often it changes. Use these values for the "data as of" label in the app.

In [9]:
def write_currency(dataset, covers_from, covers_to, snapshot_note, origin_note, frequency):
    """Record how current the data is. extracted_utc is the moment this notebook ran."""
    row = {
        "dataset": dataset,
        "extracted_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "data_covers_from": str(covers_from)[:10] if covers_from is not None else "not stated in the data",
        "data_covers_to": str(covers_to)[:10] if covers_to is not None else "not stated in the data",
        "snapshot_note": snapshot_note,
        "source_origin_note": origin_note,
        "source_update_frequency": frequency,
    }
    pd.DataFrame([row]).to_csv(OUT / "data_currency.csv", index=False, encoding="utf-8")
    for key, value in row.items():
        print(f"{key:24s} {value}")


log = json.loads(LOG_PATH.read_text(encoding="utf-8"))["benchmark_data"]
snap = ("archive downloaded in this run" if log["how"] == "downloaded"
        else "archive copied from Data/raw/epoch. That copy is the Part 2 download of 2026-10-14")
write_currency("Epoch benchmark scores (benchmark_scores.csv)",
               scores["release_date"].min(), scores["release_date"].max(), snap,
               "Dates cover the release dates of the evaluated models. The README in the archive is dated 2026-10 in its BibTeX entry. Start date of the Hub not recorded in our notes.",
               "Irregular, as new runs are added")

dataset                  Epoch benchmark scores (benchmark_scores.csv)
extracted_utc            2026-10-17T22:32:04Z
data_covers_from         2019-11-05
data_covers_to           2026-10-15
snapshot_note            archive downloaded in this run
source_origin_note       Dates cover the release dates of the evaluated models. The README in the archive is dated 2026-10 in its BibTeX entry. Start date of the Hub not recorded in our notes.
source_update_frequency  Irregular, as new runs are added


In [10]:
elapsed = time.time() - T0
print(f"Elapsed time of this run: {elapsed:.1f} seconds")
print("Files written:")
for p in sorted(OUT.glob("*")):
    print(f"  {p.relative_to(HERE)}  ({p.stat().st_size:,} bytes)")

Elapsed time of this run: 13.2 seconds
Files written:
  processed\benchmark_catalogue.csv  (6,145 bytes)
  processed\benchmark_model_metadata.csv  (122,957 bytes)
  processed\benchmark_scores.csv  (738,610 bytes)
  processed\data_currency.csv  (436 bytes)
